# Primitive Operations and Line-by-Line Cost Counting

In algorithm analysis, wall-clock timing varies across machines, compilers, and operating systems. To evaluate algorithms objectively, computer scientists use the **Random Access Machine (RAM)** model and count **primitive operations**.

## Learning Outcomes

- Understand the RAM computational model and uniform memory access.
- Identify the 7 primitive operations in Python code.
- Construct cost-and-frequency tables ($c_i \times \text{times}_i$).
- Derive exact operational polynomials $f(n)$ for straight-line code, conditionals, and loops.
- Implement instrumented step counters to empirically verify operation counts.

## 1. Wall-Clock Profiling Limitations

Measuring elapsed time using `time.perf_counter()` demonstrates why empirical wall-clock timing is noisy and machine-dependent.

In [ ]:
import time


def time_loop(iterations: int = 100_000) -> float:
    start = time.perf_counter()
    total = 0
    for i in range(iterations):
        total += i
    end = time.perf_counter()
    return end - start


runs = [time_loop() for _ in range(5)]
for idx, elapsed in enumerate(runs, 1):
    print(f'Run {idx}: {elapsed * 1000:.4f} ms')

spread = (max(runs) - min(runs)) * 1000
print(f'Timing spread: {spread:.4f} ms')


## 2. The 7 Primitive Operations in the RAM Model

The Random Access Machine (RAM) model assumes instructions execute sequentially and each primitive step requires $O(1)$ constant time:

1. **Assignment**: Assigning an identifier to an object (`x = 10`).
2. **Variable lookup**: Resolving an identifier (`x`).
3. **Arithmetic operation**: Basic math (`+`, `-`, `*`, `/`, `//`, `%`).
4. **Comparison**: Relational check (`<`, `<=`, `>`, `>=`, `==`, `!=`).
5. **Indexing**: Accessing an item by index (`arr[i]`).
6. **Function call**: Invoking a function (call dispatch overhead).
7. **Function return**: Returning a value from a function (`return v`).

In [ ]:
# Straight-line operations
number_a = 10
number_b = number_a
sum_val = number_a + number_b
is_greater = number_a > number_b

print(f'number_a = {number_a}, number_b = {number_b}')
print(f'sum_val = {sum_val}, is_greater = {is_greater}')


## 3. Counting in Conditionals

In branching code, different paths execute different numbers of operations. Worst-case analysis considers the maximum operations along any path.

In [ ]:
def find_maximum(a: float, b: float) -> float:
    """Return maximum of two numbers.
    
    Cost breakdown:
    - Call overhead: 1 op
    - Check a > b: 2 lookups + 1 comparison = 3 ops
    - Branch assign: 1 lookup + 1 assign = 2 ops
    - Return: 1 lookup + 1 return = 2 ops
    Total: 8 ops
    """
    return a if a > b else b


print('find_maximum(15, 23) ->', find_maximum(15, 23))
print('find_maximum(42, 10) ->', find_maximum(42, 10))


In [ ]:
import math


def evaluate_piecewise(a: float, b: float) -> float:
    """Piecewise function evaluating different operations per branch."""
    result = 0.0
    if a > 0:
        result = (b * b) + a
    elif a < 0:
        result = math.sqrt(b)
    return result


print('a=2, b=3   ->', evaluate_piecewise(2, 3))
print('a=-1, b=16 ->', evaluate_piecewise(-1, 16))
print('a=0, b=5   ->', evaluate_piecewise(0, 5))


## 4. Cost-Frequency Analysis for Loops

Consider a loop that increments a counter from $0$ to $n$:

```python
i = 0
while i < n:
    i = i + 1
```

| Statement | Cost | Execution Count |
| :--- | :---: | :---: |
| `i = 0` | $c_0$ | $1$ |
| `i < n` | $c_1$ | $n + 1$ |
| `i = i + 1` | $c_2$ | $n$ |

Total cost:
$$f(n) = c_0 + c_1(n + 1) + c_2 n = (c_1 + c_2)n + (c_0 + c_1)$$

This confirms $f(n)$ is a linear function $a n + b$.

In [ ]:
class InstrumentedWhileLoop:
    def __init__(self, n: int):
        self.n = n
        self.step_count = 0

    def execute(self) -> int:
        # i = 0 (1 assignment)
        self.step_count += 1
        i = 0

        # While condition evaluated (n + 1) times
        while True:
            self.step_count += 1  # test: i < n
            if i >= self.n:
                break
            # Body: i = i + 1 (lookup, add, assign -> 3 ops)
            self.step_count += 3
            i += 1

        return self.step_count


for n_val in [1, 2, 3, 4, 5, 10]:
    sim = InstrumentedWhileLoop(n_val)
    steps = sim.execute()
    expected = 4 * n_val + 2
    print(f'n = {n_val:2d} | Steps: {steps:2d} | Expected (4n + 2): {expected:2d}')
    assert steps == expected


## 5. Verification Exercises

Trace and count operations for an accumulator function.

In [ ]:
def instrumented_sum(n: int) -> tuple[int, int]:
    """Accumulate integers 0..n-1 with step accounting."""
    steps = 0

    total = 0
    steps += 1  # total = 0

    i = 0
    steps += 1  # i = 0

    while True:
        steps += 1  # condition i < n
        if i >= n:
            break
        total += i
        steps += 4  # lookup total, lookup i, add, assign
        i += 1
        steps += 3  # lookup i, add 1, assign

    steps += 2  # return lookup + return
    return total, steps


result, step_total = instrumented_sum(10)
print(f'Sum: {result}, Steps: {step_total}')
assert result == 45
assert step_total == 85  # 2 + 11 + 7*10 + 2 = 85
print('All step assertions passed successfully!')
